In [1]:
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Charger les données préparées :
df = pd.read_csv('donnees_meteo_2025_NETTOYEES.csv', sep=';')

# Sélectionner les variables explicatives (features) et la cible.
# On exclut 'station_id' et 'dh_utc' qui ne sont pas des métriques météo.
X = df.drop(['station_id', 'dh_utc', 'pluie_1h_mm', 'pleut_il'], axis=1)
y = df['pleut_il']  # Notre cible binaire

# Séparer les données en entraînement (80%) et test (20%) :
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Créer et entraîner le modèle XGBoost
model = xgb.XGBClassifier(
    objective='binary:logistic',  # Classification binaire
    n_estimators=100,            # Nombre d'arbres
    max_depth=8,                 # Profondeur maximale des arbres
    learning_rate=0.3,           # Vitesse d'apprentissage
    random_state=42
)

print("Entraînement du modèle...")

# Entraînement du modèle sur les données d'apprentissage :
model.fit(X_train, y_train)

# Génération des prédictions sur le jeu de test :
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Faire des prédictions sur l'ensemble de test :
y_pred = model.predict(X_test)

# Évaluer les performances :
accuracy = accuracy_score(y_test, y_pred)
print(f"\nPrécision du modèle : {accuracy:.2f}")
print("\nRapport de classification :")
print(classification_report(y_test, y_pred))

rmse = np.sqrt(mean_squared_error(y_test, y_proba))
mae = mean_absolute_error(y_test, y_proba)
print(f"\nRMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

# Sauvegarder le modèle pour une utilisation future
model.save_model('modele_prediction_pluie.json')
print("\nModèle sauvegardé sous 'modele_prediction_pluie.json'")

Entraînement du modèle...

Précision du modèle : 0.95

Rapport de classification :
              precision    recall  f1-score   support

           0       0.96      0.99      0.97      3312
           1       0.63      0.36      0.46       198

    accuracy                           0.95      3510
   macro avg       0.80      0.68      0.72      3510
weighted avg       0.94      0.95      0.95      3510


RMSE : 0.1943
MAE  : 0.0607

Modèle sauvegardé sous 'modele_prediction_pluie.json'
